In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle

In [3]:
#load the model,scaler,one hot encoder pickle

model = load_model("model.h5")

#scaler
with open("label_encoder_gender.pkl","rb") as file:
    label_encoder_gender = pickle.load(file)

#ohe
with open("one_hot_encoder_geo.pkl","rb") as file:
    oneHotEncoder_geo = pickle.load(file)
    
#scaler
with open("scaler.pkl","rb") as file:
    scaler = pickle.load(file)

In [ ]:
# Prepare the input data
input_data = {
    'CreditScore':500,
    'Geography':'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure':3 ,
    'Balance': 6000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [5]:
##INPUT DATA TO DATAFRAME

input_df = pd.DataFrame([input_data])
input_df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,500,France,Male,40,3,6000,2,1,1,50000


In [6]:
#label encodign to gender 
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [ ]:
##onehotencoding to geography
# since sparse_output = false is used before no need input_df[['Geography']].to array 
geo_encoded = oneHotEncoder_geo.transform(input_df[['Geography']]) 

geo_df = pd.DataFrame(
    geo_encoded,
    columns=oneHotEncoder_geo.get_feature_names_out(['Geography'])
)


In [17]:
input_df = input_df.drop('Geography', axis=1)

input_df = pd.concat([input_df.reset_index(drop=True),
                      geo_df.reset_index(drop=True)], axis=1)
input_df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,500,1,40,3,6000,2,1,1,50000,1.0,0.0,0.0


In [18]:
#scaling 
input_scaled = scaler.transform(input_df)
input_scaled

array([[-1.57375827,  0.91324755,  0.10479359, -0.69539349, -1.12240462,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [19]:
#predict churn 

prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 841ms/step


array([[0.01304953]], dtype=float32)

In [20]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.01304953)

In [21]:
if prediction_proba > 0.5:
    print("customer is likely to churn")
else:
    print("customer is not likely to churn")

customer is not likely to churn
